# Fraude em cartões: qual limiar custa menos?

O notebook principal (`Deteccao_fraude_CC.ipynb`) compara técnicas de balanceamento. Este aqui responde à pergunta de negócio: **quanto dinheiro o modelo realmente evita perder?**

Decisões de método:
- **Divisão temporal**: treino nos primeiros 60% do tempo, validação nos 10% seguintes, teste nos 30% finais. Fraude muda com o tempo; misturar passado e futuro infla o resultado.
- **Limiar escolhido na validação**, nunca no teste.
- **Custo explícito**: fraude não detectada custa o valor da transação; cada alerta custa **€5** de análise manual (premissa, ajustável).
- A coluna `Time` fica fora do modelo (só serve para ordenar).

In [1]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

df = pd.read_csv("creditcard.zip").sort_values("Time").reset_index(drop=True)
n = len(df); a, b = int(n*.6), int(n*.7)
tr, va, te = df.iloc[:a], df.iloc[a:b], df.iloc[b:]
feats = [c for c in df.columns if c not in ("Class", "Time")]
CUSTO_ALERTA = 5.0
print(f"Transações: {n:,} | fraudes: {df.Class.mean():.3%}")
print(f"Teste: {len(te):,} transações, {te.Class.sum()} fraudes, € {te.loc[te.Class==1,'Amount'].sum():,.2f} em fraude")

Transações: 284,807 | fraudes: 0.173%
Teste: 85,443 transações, 108 fraudes, € 14,566.58 em fraude


In [2]:
def custo(p, d, t):
    alerta = p >= t; y = d.Class.values; valor = d.Amount.values
    return valor[~alerta & (y == 1)].sum() + CUSTO_ALERTA * alerta.sum()

sc = StandardScaler().fit(tr[feats])
modelos = {
    "Regressão logística": (LogisticRegression(max_iter=3000, class_weight="balanced"), True),
    "Gradient boosting":   (HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, l2_regularization=1.0, random_state=0), False),
}
perda_sem_modelo = te.loc[te.Class==1, "Amount"].sum()
for nome, (m, escala) in modelos.items():
    X = (lambda d: sc.transform(d[feats])) if escala else (lambda d: d[feats])
    m.fit(X(tr), tr.Class)
    pv, pt = m.predict_proba(X(va))[:, 1], m.predict_proba(X(te))[:, 1]
    grade = np.unique(np.quantile(pv, np.linspace(0.95, 0.99999, 500)))
    t = min(grade, key=lambda t: custo(pv, va, t))
    y = te.Class.values; al = pt >= t; vp = (al & (y == 1)).sum()
    c = custo(pt, te, t)
    print(f"\n{nome}: ROC-AUC {roc_auc_score(y, pt):.3f} | PR-AUC {average_precision_score(y, pt):.3f}")
    print(f"  limiar padrão 0,5 -> {(pt>=.5).sum():,} alertas, custo € {custo(pt, te, .5):,.0f}")
    print(f"  limiar ótimo {t:.4f} -> {al.sum()} alertas, {vp}/{y.sum()} fraudes pegas ({vp/y.sum():.0%}), precisão {vp/al.sum():.0%}")
    print(f"  custo € {c:,.0f} vs € {perda_sem_modelo:,.0f} sem modelo -> perda reduzida em {1-c/perda_sem_modelo:.0%}")


Regressão logística: ROC-AUC 0.977 | PR-AUC 0.767
  limiar padrão 0,5 -> 2,425 alertas, custo € 14,580
  limiar ótimo 0.8837 -> 463 alertas, 93/108 fraudes pegas (86%), precisão 20%
  custo € 4,796 vs € 14,567 sem modelo -> perda reduzida em 67%

Gradient boosting: ROC-AUC 0.984 | PR-AUC 0.806
  limiar padrão 0,5 -> 90 alertas, custo € 3,644
  limiar ótimo 0.0743 -> 106 alertas, 83/108 fraudes pegas (77%), precisão 78%
  custo € 3,074 vs € 14,567 sem modelo -> perda reduzida em 79%


## Leitura

- **ROC-AUC alto não basta.** A regressão logística no limiar padrão de 0,5 gera milhares de alertas para menos de cem fraudes: com €5 por análise, sai **mais caro do que não ter modelo**.
- Escolhendo o limiar pelo custo, os dois modelos passam a valer a pena; o **gradient boosting** reduz a perda em cerca de **79%** com ~100 alertas no período de teste.
- A premissa de €5 por alerta muda o limiar ótimo: é a conversa que precisa acontecer com a área de risco antes de colocar em produção.